In [1]:
from pathlib import Path
import zipfile
import pandas as pd
import geopandas as gpd
import gc
import numpy as np
import matplotlib.pyplot as plt
import yaml

#### 人口データの結合

In [2]:
# ----------------------------
# config + fallback
# ----------------------------
CONFIG_PATH = Path("config.yaml")

if CONFIG_PATH.exists():
    with open(CONFIG_PATH, "r", encoding="utf-8") as f:
        config = yaml.safe_load(f)

    DATA_DIR = Path(config["data_root"]).expanduser()

    if "project_root" in config:
        PROJECTS_DIR = Path(config["project_root"]).expanduser()
    else:
        PROJECTS_DIR = DATA_DIR.parent / "Projects"
else:
    DATA_DIR = Path("~/Desktop/Research/Data").expanduser()
    PROJECTS_DIR = Path("~/Desktop/Research/Projects").expanduser()

RESEARCH_DIR = DATA_DIR.parent

print("RESEARCH_DIR:", RESEARCH_DIR)
print("DATA_DIR:", DATA_DIR)
print("PROJECTS_DIR:", PROJECTS_DIR)

# ----------------------------
# LandUse
# ----------------------------
LANDUSE_RAW_DIR = DATA_DIR / "Raw" / "LandUse" / "2021" / "national_landuse"
LANDUSE_INTERMEDIATE_DIR = DATA_DIR / "Intermediate" / "LandUse" / "2021"
LANDUSE_EXTRACT_DIR = LANDUSE_INTERMEDIATE_DIR / "extracted" / "landuse_2021"
LANDUSE_PROCESSED_DIR = DATA_DIR / "Processed" / "LandUse" / "2021"

# ----------------------------
# ZIP polygon (すでに展開済みshapeを使う想定)
# ----------------------------
ZIP_RAW_DIR = DATA_DIR / "Raw" / "Geospatial" / "ZIP_polygon" / "2022" / "2022_GEOSPACE郵便番号ポリゴン"
ZIP_SHAPE_DIR = ZIP_RAW_DIR / "shape"
ZIP_SHP_PATH = ZIP_SHAPE_DIR / "GSP99.shp"
ZIP_EXTRACT_DIR = LANDUSE_INTERMEDIATE_DIR / "extracted" / "zipcode_polygon_2022"

# ----------------------------
# 500m mesh
# Raw 側に「内側 zip 群」がすでに置かれている構造
# ----------------------------
MESH_RAW_DIR = DATA_DIR / "Raw" / "Geospatial" / "Mesh" / "500m_2024"
MESH_RAW_INNER_DIR = MESH_RAW_DIR / "500m_mesh_2024_GEOJSON"

MESH_EXTRACT_DIR = LANDUSE_INTERMEDIATE_DIR / "extracted" / "mesh_500m_2024"
MESH_GEOJSON_DIR = MESH_EXTRACT_DIR / "geojson"

# ----------------------------
# Project output
# ----------------------------
LANDUSE_BUILD_DIR = PROJECTS_DIR / "LandUse_build"
LANDUSE_MAP_DIR = LANDUSE_BUILD_DIR / "output" / "maps"

for p in [
    LANDUSE_EXTRACT_DIR,
    LANDUSE_PROCESSED_DIR,
    ZIP_EXTRACT_DIR,
    MESH_EXTRACT_DIR,
    MESH_GEOJSON_DIR,
    LANDUSE_MAP_DIR,
]:
    p.mkdir(parents=True, exist_ok=True)

# notebook内で使う短い別名
PROCESSED_DIR = LANDUSE_PROCESSED_DIR
csv_path = PROCESSED_DIR / "zipcode_landuse_area.csv"
gpkg_path = PROCESSED_DIR / "zipcode_landuse_area.gpkg"

print("MESH_RAW_INNER_DIR:", MESH_RAW_INNER_DIR)
print("  exists:", MESH_RAW_INNER_DIR.exists())

print("\nSample mesh zip files:")
for f in sorted(MESH_RAW_INNER_DIR.glob("*.zip"))[:5]:
    print(" ", f.name)

print("\nZIP_SHP_PATH:", ZIP_SHP_PATH)
print("  exists:", ZIP_SHP_PATH.exists())

RESEARCH_DIR: /Users/Tomo/Desktop/Research
DATA_DIR: /Users/Tomo/Desktop/Research/Data
PROJECTS_DIR: /Users/Tomo/Desktop/Research/Projects
MESH_RAW_INNER_DIR: /Users/Tomo/Desktop/Research/Data/Raw/Geospatial/Mesh/500m_2024/500m_mesh_2024_GEOJSON
  exists: True

Sample mesh zip files:
  500m_mesh_2024_01_GEOJSON.zip
  500m_mesh_2024_02_GEOJSON.zip
  500m_mesh_2024_03_GEOJSON.zip
  500m_mesh_2024_04_GEOJSON.zip
  500m_mesh_2024_05_GEOJSON.zip

ZIP_SHP_PATH: /Users/Tomo/Desktop/Research/Data/Raw/Geospatial/ZIP_polygon/2022/2022_GEOSPACE郵便番号ポリゴン/shape/GSP99.shp
  exists: True


In [3]:
mesh_raw_dir = MESH_RAW_INNER_DIR
mesh_extract_dir = MESH_EXTRACT_DIR
mesh_geojson_dir = MESH_GEOJSON_DIR

# Raw側にある内側zip一覧
inner_zip_files = sorted(mesh_raw_dir.glob("*.zip"))
print("Inner mesh zip files found:", len(inner_zip_files))
for f in inner_zip_files[:5]:
    print(" ", f.name)

# まだgeojsonが1つも無ければ解凍
existing_geojson = sorted(mesh_geojson_dir.rglob("*.geojson"))

if len(existing_geojson) == 0:
    for zip_path in inner_zip_files:
        out_dir = mesh_geojson_dir / zip_path.stem
        out_dir.mkdir(parents=True, exist_ok=True)

        # すでにこのzip由来のgeojsonが存在する場合はスキップ
        if any(out_dir.rglob("*.geojson")):
            continue

        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(out_dir)

    print("Inner unzip done.")
else:
    print("GeoJSON already exists. Skip unzip.")

# geojson一覧取得
geojson_files = sorted(mesh_geojson_dir.rglob("*.geojson"))

print("geojson files:", len(geojson_files))
print(geojson_files[:5])

Inner mesh zip files found: 47
  500m_mesh_2024_01_GEOJSON.zip
  500m_mesh_2024_02_GEOJSON.zip
  500m_mesh_2024_03_GEOJSON.zip
  500m_mesh_2024_04_GEOJSON.zip
  500m_mesh_2024_05_GEOJSON.zip
GeoJSON already exists. Skip unzip.
geojson files: 47
[PosixPath('/Users/Tomo/Desktop/Research/Data/Intermediate/LandUse/2021/extracted/mesh_500m_2024/geojson/500m_mesh_2024_01_GEOJSON/500m_mesh_2024_01_GEOJSON/500m_mesh_2024_01.geojson'), PosixPath('/Users/Tomo/Desktop/Research/Data/Intermediate/LandUse/2021/extracted/mesh_500m_2024/geojson/500m_mesh_2024_02_GEOJSON/500m_mesh_2024_02_GEOJSON/500m_mesh_2024_02.geojson'), PosixPath('/Users/Tomo/Desktop/Research/Data/Intermediate/LandUse/2021/extracted/mesh_500m_2024/geojson/500m_mesh_2024_03_GEOJSON/500m_mesh_2024_03_GEOJSON/500m_mesh_2024_03.geojson'), PosixPath('/Users/Tomo/Desktop/Research/Data/Intermediate/LandUse/2021/extracted/mesh_500m_2024/geojson/500m_mesh_2024_04_GEOJSON/500m_mesh_2024_04_GEOJSON/500m_mesh_2024_04.geojson'), PosixPath('/Us

#### 人口データを各郵便番号ポリゴンに按分で入れていく

##### 郵便番号ポリゴンと既存テーブルを読む

In [4]:
import gc
import pandas as pd
import geopandas as gpd

mesh_files = sorted(MESH_GEOJSON_DIR.rglob("*.geojson"))
print("mesh files:", len(mesh_files))
print(mesh_files[:5])

zipcode_area = pd.read_csv(csv_path, dtype={"zipcode": str})
zipcode_gdf = gpd.read_file(gpkg_path)

zipcode_area["zipcode"] = zipcode_area["zipcode"].astype(str).str.zfill(7)
zipcode_gdf["zipcode"] = zipcode_gdf["zipcode"].astype(str).str.zfill(7)

# 必要列だけ
zip_base = zipcode_gdf[["zipcode", "geometry"]].copy()

print(zip_base.shape)
print(zip_base.crs)

mesh files: 47
[PosixPath('/Users/Tomo/Desktop/Research/Data/Intermediate/LandUse/2021/extracted/mesh_500m_2024/geojson/500m_mesh_2024_01_GEOJSON/500m_mesh_2024_01_GEOJSON/500m_mesh_2024_01.geojson'), PosixPath('/Users/Tomo/Desktop/Research/Data/Intermediate/LandUse/2021/extracted/mesh_500m_2024/geojson/500m_mesh_2024_02_GEOJSON/500m_mesh_2024_02_GEOJSON/500m_mesh_2024_02.geojson'), PosixPath('/Users/Tomo/Desktop/Research/Data/Intermediate/LandUse/2021/extracted/mesh_500m_2024/geojson/500m_mesh_2024_03_GEOJSON/500m_mesh_2024_03_GEOJSON/500m_mesh_2024_03.geojson'), PosixPath('/Users/Tomo/Desktop/Research/Data/Intermediate/LandUse/2021/extracted/mesh_500m_2024/geojson/500m_mesh_2024_04_GEOJSON/500m_mesh_2024_04_GEOJSON/500m_mesh_2024_04.geojson'), PosixPath('/Users/Tomo/Desktop/Research/Data/Intermediate/LandUse/2021/extracted/mesh_500m_2024/geojson/500m_mesh_2024_05_GEOJSON/500m_mesh_2024_05_GEOJSON/500m_mesh_2024_05.geojson')]
(113186, 2)
EPSG:6668


##### 面積計算用の投影系

In [5]:
JAPAN_AEA = (
    "+proj=aea +lat_1=30 +lat_2=46 +lat_0=36 +lon_0=138 "
    "+x_0=0 +y_0=0 +datum=WGS84 +units=m +no_defs"
)

##### 各 GeoJSON を1つずつ読み、郵便番号と重ねて人口を按分

In [6]:
pop_results = []

for i, mesh_file in enumerate(mesh_files, 1):
    print(f"[{i}/{len(mesh_files)}] {mesh_file.name}")

    mesh_gdf = gpd.read_file(mesh_file)

    # 必要列だけ
    mesh_gdf = mesh_gdf[["MESH_ID", "PTN_2020", "geometry"]].copy()

    # 人口を数値化
    mesh_gdf["PTN_2020"] = pd.to_numeric(mesh_gdf["PTN_2020"], errors="coerce").fillna(0)

    # 人口ゼロメッシュを落とす
    mesh_gdf = mesh_gdf[mesh_gdf["PTN_2020"] > 0].copy()
    if mesh_gdf.empty:
        continue

    # CRSを揃える
    if mesh_gdf.crs != zip_base.crs:
        mesh_gdf = mesh_gdf.to_crs(zip_base.crs)

    # zip側を bbox で絞る
    xmin, ymin, xmax, ymax = mesh_gdf.total_bounds
    zip_sub = zip_base.cx[xmin:xmax, ymin:ymax].copy()

    if zip_sub.empty:
        continue

    # 投影変換
    mesh_sub = mesh_gdf.to_crs(JAPAN_AEA)
    zip_sub = zip_sub.to_crs(JAPAN_AEA)

    # メッシュ面積
    mesh_sub["mesh_area_m2"] = mesh_sub.geometry.area

    # overlay
    inter = gpd.overlay(
        mesh_sub[["MESH_ID", "PTN_2020", "mesh_area_m2", "geometry"]],
        zip_sub[["zipcode", "geometry"]],
        how="intersection",
        keep_geom_type=False
    )

    if inter.empty:
        continue

    # 交差面積
    inter["intersect_area_m2"] = inter.geometry.area

    # 面積按分人口
    inter["pop_alloc"] = inter["PTN_2020"] * (
        inter["intersect_area_m2"] / inter["mesh_area_m2"]
    )

    # 郵便番号ごとに合計
    pop_by_zip = (
        inter.groupby("zipcode", as_index=False)["pop_alloc"]
        .sum()
        .rename(columns={"pop_alloc": "population_2020"})
    )

    pop_results.append(pop_by_zip)

    del mesh_gdf, mesh_sub, zip_sub, inter, pop_by_zip
    gc.collect()

[1/47] 500m_mesh_2024_01.geojson
[2/47] 500m_mesh_2024_02.geojson
[3/47] 500m_mesh_2024_03.geojson
[4/47] 500m_mesh_2024_04.geojson
[5/47] 500m_mesh_2024_05.geojson
[6/47] 500m_mesh_2024_06.geojson
[7/47] 500m_mesh_2024_07.geojson
[8/47] 500m_mesh_2024_08.geojson
[9/47] 500m_mesh_2024_09.geojson
[10/47] 500m_mesh_2024_10.geojson
[11/47] 500m_mesh_2024_11.geojson
[12/47] 500m_mesh_2024_12.geojson
[13/47] 500m_mesh_2024_13.geojson
[14/47] 500m_mesh_2024_14.geojson
[15/47] 500m_mesh_2024_15.geojson
[16/47] 500m_mesh_2024_16.geojson
[17/47] 500m_mesh_2024_17.geojson
[18/47] 500m_mesh_2024_18.geojson
[19/47] 500m_mesh_2024_19.geojson
[20/47] 500m_mesh_2024_20.geojson
[21/47] 500m_mesh_2024_21.geojson
[22/47] 500m_mesh_2024_22.geojson
[23/47] 500m_mesh_2024_23.geojson
[24/47] 500m_mesh_2024_24.geojson
[25/47] 500m_mesh_2024_25.geojson
[26/47] 500m_mesh_2024_26.geojson
[27/47] 500m_mesh_2024_27.geojson
[28/47] 500m_mesh_2024_28.geojson
[29/47] 500m_mesh_2024_29.geojson
[30/47] 500m_mesh_2024_

##### 全ファイルの結果を結合

In [7]:
zipcode_pop = pd.concat(pop_results, ignore_index=True)

zipcode_pop = (
    zipcode_pop.groupby("zipcode", as_index=False)["population_2020"]
    .sum()
)

print(zipcode_pop.shape)
zipcode_pop.head()

(112587, 2)


,zipcode,population_2020
0,000None,462.351454
1,0010010,987.165864
2,0010011,992.571984
3,0010012,1056.624154
4,0010013,1062.163177


#### 既存の zipcode_area に人口を結合

In [8]:
zipcode_area = zipcode_area.drop(columns=["population_2020"], errors="ignore")

zipcode_area = zipcode_area.merge(
    zipcode_pop,
    on="zipcode",
    how="left"
)

zipcode_area["population_2020"] = zipcode_area["population_2020"].fillna(0)

##### 人口密度を計算

In [9]:
# 0除算回避
habitable_km2 = zipcode_area["habitable_area_m2"] / 1_000_000
residential_km2 = zipcode_area["residential_area_m2"] / 1_000_000

zipcode_area["pop_density_habitable"] = np.where(
    habitable_km2 > 0,
    zipcode_area["population_2020"] / habitable_km2,
    np.nan
)

zipcode_area["pop_density_residential"] = np.where(
    residential_km2 > 0,
    zipcode_area["population_2020"] / residential_km2,
    np.nan
)

zipcode_area["pop_density_habitable"] = zipcode_area["pop_density_habitable"].round(6)
zipcode_area["pop_density_residential"] = zipcode_area["pop_density_residential"].round(6)

##### 保存

In [10]:
zipcode_area.to_csv(csv_path, index=False, encoding="utf-8-sig")
print("Overwritten CSV:", csv_path)

print("\nCSV columns after save:")
print(zipcode_area.columns.tolist())

zipcode_gdf = zipcode_gdf.drop(
    columns=["population_2020", "pop_density_habitable", "pop_density_residential"],
    errors="ignore"
)

zipcode_gdf = zipcode_gdf.merge(
    zipcode_area[[
        "zipcode",
        "population_2020",
        "pop_density_habitable",
        "pop_density_residential"
    ]],
    on="zipcode",
    how="left"
)

print("\nGPKG columns before save:")
print(zipcode_gdf.columns.tolist())

zipcode_gdf.to_file(gpkg_path, driver="GPKG")
print("Overwritten GPKG:", gpkg_path)

# 念のため再読込して確認
check_gdf = gpd.read_file(gpkg_path)
print("\nGPKG columns after re-read:")
print(check_gdf.columns.tolist())

Overwritten CSV: /Users/Tomo/Desktop/Research/Data/Processed/LandUse/2021/zipcode_landuse_area.csv

CSV columns after save:
['zipcode', 'total_area_m2', 'forest_area_m2', 'building_area_m2', 'water_area_m2', 'beach_area_m2', 'sea_area_m2', 'habitable_area_m2', 'residential_area_m2', 'habitable_ratio', 'residential_ratio', 'population_2020', 'pop_density_habitable', 'pop_density_residential']

GPKG columns before save:
['zipcode', 'prefcode', 'total_area_m2', 'forest_area_m2', 'building_area_m2', 'water_area_m2', 'beach_area_m2', 'sea_area_m2', 'habitable_area_m2', 'residential_area_m2', 'habitable_ratio', 'residential_ratio', 'geometry', 'population_2020', 'pop_density_habitable', 'pop_density_residential']
Overwritten GPKG: /Users/Tomo/Desktop/Research/Data/Processed/LandUse/2021/zipcode_landuse_area.gpkg

GPKG columns after re-read:
['zipcode', 'prefcode', 'total_area_m2', 'forest_area_m2', 'building_area_m2', 'water_area_m2', 'beach_area_m2', 'sea_area_m2', 'habitable_area_m2', 'res